<a href="https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-2-generative-ai/lab-04-architecture-trace.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 4 (graded) — Architecture trace
**Course 2: Generative AI and LLMs with Python — Chapter 4: Modern LLM architecture**

**What you'll submit:** RoPE implemented and unit-tested, a swap of RoPE into Chapter 3's
model with a before/after comparison, and an annotated walkthrough of a real 2024–25 open
model's config mapping every value to a concept from the chapter.

## 1. RoPE, from scratch
Fill in the `TODO`s: rotate each (even, odd) pair of dimensions in Q/K by an angle that
scales with position `m` and with the pair's frequency.

In [ ]:
import torch
import math

torch.manual_seed(0)

def build_rope_cache(seq_len, dim, base=10000.0):
    """Returns cos, sin of shape (seq_len, dim/2), precomputed once per sequence length."""
    assert dim % 2 == 0
    # TODO: inv_freq[i] = 1 / base^(2i/dim) for i in [0, dim/2)
    inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
    positions = torch.arange(seq_len).float()
    angles = torch.outer(positions, inv_freq)  # (seq_len, dim/2) — position m times frequency i
    return torch.cos(angles), torch.sin(angles)

def apply_rope(x, cos, sin):
    """x: (..., seq_len, dim). Rotates each consecutive pair of dims by the cached angle."""
    x1, x2 = x[..., 0::2], x[..., 1::2]  # even, odd dims
    # TODO: standard 2D rotation:
    #   x1' = x1 * cos - x2 * sin
    #   x2' = x1 * sin + x2 * cos
    x1_rot = x1 * cos - x2 * sin
    x2_rot = x1 * sin + x2 * cos
    out = torch.stack([x1_rot, x2_rot], dim=-1).flatten(-2)
    return out

## 2. Unit tests — run before continuing

In [ ]:
def test_rope():
    seq_len, dim = 16, 8
    cos, sin = build_rope_cache(seq_len, dim)
    x = torch.randn(2, seq_len, dim)  # (batch, seq, dim)
    out = apply_rope(x, cos.unsqueeze(0), sin.unsqueeze(0))
    assert out.shape == x.shape, f'shape changed: {x.shape} -> {out.shape}'

    # rotation preserves vector norm at each position
    assert torch.allclose(x.norm(dim=-1), out.norm(dim=-1), atol=1e-4), \
        'RoPE should preserve the norm of each vector (it is a rotation)'

    # position 0 should be unrotated (cos=1, sin=0 there)
    assert torch.allclose(x[:, 0, :], out[:, 0, :], atol=1e-4), \
        'position 0 should be unchanged by RoPE (angle = 0)'

    # the KEY property: dot product between two rotated vectors depends only on their
    # RELATIVE distance, not their absolute positions
    q = torch.randn(1, 1, dim)
    k = torch.randn(1, 1, dim)
    q_at_5 = apply_rope(q, cos[5:6].unsqueeze(0), sin[5:6].unsqueeze(0))
    k_at_8 = apply_rope(k, cos[8:9].unsqueeze(0), sin[8:9].unsqueeze(0))
    q_at_10 = apply_rope(q, cos[10:11].unsqueeze(0), sin[10:11].unsqueeze(0))
    k_at_13 = apply_rope(k, cos[13:14].unsqueeze(0), sin[13:14].unsqueeze(0))
    dot_5_8 = (q_at_5 * k_at_8).sum()
    dot_10_13 = (q_at_10 * k_at_13).sum()  # same relative distance (3), different absolute positions
    assert torch.allclose(dot_5_8, dot_10_13, atol=1e-3), \
        'dot product should depend on relative distance only — got different values at the same offset'

    print('All RoPE tests PASSED.')

test_rope()

## 3. Swap RoPE into a small attention block and compare

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class RoPEAttention(nn.Module):
    def __init__(self, d_model, n_heads, max_seq_len=64):
        super().__init__()
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)
        cos, sin = build_rope_cache(max_seq_len, self.d_k)
        self.register_buffer('cos', cos)
        self.register_buffer('sin', sin)

    def forward(self, x):
        B, S, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q = q.view(B, S, self.n_heads, self.d_k).transpose(1, 2)
        k = k.view(B, S, self.n_heads, self.d_k).transpose(1, 2)
        v = v.view(B, S, self.n_heads, self.d_k).transpose(1, 2)
        q = apply_rope(q, self.cos[:S], self.sin[:S])
        k = apply_rope(k, self.cos[:S], self.sin[:S])
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_k)
        out = (F.softmax(scores, dim=-1) @ v).transpose(1, 2).reshape(B, S, C)
        return self.proj(out)

x = torch.randn(2, 20, 32)
rope_attn = RoPEAttention(d_model=32, n_heads=4)
out = rope_attn(x)
print('RoPE attention output shape:', out.shape)
print('Chapter 1 used learned/sinusoidal absolute positions; this block instead rotates')
print('Q/K directly, so relative position is baked into every dot product — see the test above.')

## 4. Trace a real 2024-25 open model's config

In [ ]:
!pip install -q transformers

In [ ]:
def load_model_config(model_id='Qwen/Qwen2.5-0.5B'):
    try:
        from transformers import AutoConfig
        cfg = AutoConfig.from_pretrained(model_id)
        print(f'Loaded the real config for {model_id}.')
        return cfg.to_dict()
    except Exception as e:
        print(f'Offline fallback engaged ({e}) — using Qwen2.5-0.5B\'s publicly documented config values.')
        return {
            'hidden_size': 896, 'num_hidden_layers': 24, 'num_attention_heads': 14,
            'num_key_value_heads': 2, 'intermediate_size': 4864, 'rope_theta': 1000000.0,
            'max_position_embeddings': 32768, 'vocab_size': 151936, 'hidden_act': 'silu',
            'rms_norm_eps': 1e-06,
        }

config = load_model_config()
for k, v in config.items():
    print(f'{k:28s}: {v}')

## 5. Annotated walkthrough (fill in)
Map each config value below to a Chapter 4 concept:
- `hidden_size`, `num_hidden_layers`, `num_attention_heads`: _fill in — the model's overall shape_
- `num_key_value_heads` vs `num_attention_heads`: _fill in — this is GQA; compute the ratio,
  and say what it implies about KV-cache size vs. a model with `num_key_value_heads ==
  num_attention_heads`_
- `rope_theta` and `max_position_embeddings`: _fill in — this is the RoPE base you
  implemented above, and the context length it was designed for_
- `hidden_act` (e.g. `silu`): _fill in — this is the gated-MLP activation (SwiGLU family)_
- `rms_norm_eps`: _fill in — RMSNorm, not LayerNorm; what's the practical difference?_

_Your answer here._

---
*Beacon AI · AIBits Academy — Chapter 4: Modern LLM architecture*